#Notebook 04: Silver to Gold - Advanced Transformations

**Exam Coverage**: Section 3 (Incremental Data Processing)

**Duration**: 60-75 minutes

---
## Learning Objectives
By the end of this notebook, you will be able to:
- Create business-level aggregations for reporting
- Apply window functions for advanced analytics
- Calculate customer lifetime value (LTV) with RFM segmentation
- Implement product performance ranking
- Build funnel analysis from event data
- Optimize Gold tables with Z-ordering and partitioning
---

## Section 1: Introduction to Gold Layer
The Gold layer contains **business-level aggregations** optimized for:
- **BI Reports**: Pre-aggregated metrics for dashboards
- **Machine Learning**: Feature tables for ML models
- **Executive Analytics**: KPIs and business metrics
### 🥇 Gold Layer Characteristics
| Aspect | Approach |
|--------|----------|
| **Structure** | Denormalized (optimized for queries) |
| **Calculations** | Pre-aggregated (computed once) |
| **Partitioning** | Time-series optimized |
| **Indexing** | Z-ordered for co-location |
| **Names** | Business-friendly terminology |
### Common Gold Patterns
1. **Time-series aggregations**: Daily/weekly/monthly summaries
2. **Snapshot tables**: Point-in-time state capture
3. **Dimension tables**: Enriched, slowly changing
4. **Metric tables**: Pre-calculated KPIs

In [0]:
%run ./variables

# Configuration Variables

Central configuration file for the Databricks Data Engineer Certification Lab.

**Usage**: Import this file in all notebooks to maintain consistent naming.

```python
%run ./variables
```

## Unity Catalog Configuration

## Volume Paths

## Checkpoint Locations

## Table Names

...

## Browser Types

## Locations (US Cities)

## Helper Functions

## Validation

## Display Configuration Summary

In [0]:
# Set current catalog and schema to Gold
spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"USE SCHEMA {GOLD_SCHEMA}")

print(f"Current Catalog: {spark.catalog.currentCatalog()}")
print(f"Current Schema: {spark.catalog.currentDatabase()}")

Current Catalog: cert_prep_catalog
Current Schema: `03_gold`


In [0]:
# Import required functions
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

## Section 2: Daily Sales Summary with Rolling Windows
Create a daily sales summary for business dashboards.
### Metrics to Calculate
- Total sales by day and category
- Revenue, order counts, unique customers
- Average order value
- Rolling 7-day and 30-day averages
### Window Functions for Trends
**Rolling windows** calculate moving averages:
```python
Window.partitionBy("category").orderBy("date").rowsBetween(-6, 0)  # 7 days
```
This captures the current row plus 6 preceding rows.

In [0]:
# Load Silver sales data
sales_silver = spark.table(SALES_SILVER_TABLE)

print(f"Total sales records: {sales_silver.count():,}")
sales_silver.printSchema()

Total sales records: 101,919
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- subtotal: decimal(21,2) (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_timestamp: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- processed_at: timestamp (nullable = true)
 |-- is_valid_sale: boolean (nullable = true)



---
### 🎯 EXERCISE 1: Create Daily Sales Summary
**Your task**: Aggregate sales data by date and category.
**Requirements:**
1. Add `sale_date` column (convert `order_timestamp` to date)
2. Group by `sale_date` and `category`
3. Calculate aggregations:
  - `total_revenue`: Sum of total_amount
  - `order_count`: Count of order_id
  - `unique_customers`: Count distinct customer_id
  - `avg_order_value`: Average of total_amount
  - `total_units_sold`: Sum of quantity
4. Add calculated columns:
  - `revenue_per_customer`: total_revenue / unique_customers
  - `units_per_order`: total_units_sold / order_count

**Functions needed:**
```python
F.to_date("timestamp_col")
F.sum(), F.count(), F.countDistinct(), F.avg()
```

**Hint**: Use `.withColumn()` to add sale_date, then `.groupBy().agg()`

# TODO: Calculate RFM scores using ntile

customer_ltv_rfm = customer_ltv \
    .withColumn("recency_score", 
                # TODO: 6 minus ntile of days_since_last_purchase (desc)
                6 - F.ntile(5).over(Window.orderBy(F.desc(F.col("days_since_last_purchase"))))) \
    .withColumn("frequency_score",
                # TODO: ntile of total_orders
                F.ntile(5).over(Window.orderBy(F.col("total_orders")))) \
    .withColumn("monetary_score",
                # TODO: ntile of total_spend
                F.ntile(5).over(Window.orderBy(F.col("total_spend"))))

# TODO: Calculate combined RFM score
customer_ltv_rfm = customer_ltv_rfm \
    .withColumn("rfm_score",   # TODO: sum of R + F + M
                F.col("recency_score") + F.col("frequency_score") + F.col("monetary_score"))

display(customer_ltv_rfm.orderBy(F.desc("rfm_score")).limit(20))

In [0]:
# TODO: Create daily sales summary

daily_sales_summary = sales_silver \
    .withColumn("sale_date", F.to_date(F.col("order_timestamp"))) \
    .groupBy(F.col("sale_date"), F.col("category")) \
    .agg(
        # TODO: Add aggregation functions
        F.sum(F.col("subtotal")).alias("total_revenue"),
        F.count(F.col("order_id")).alias("order_count"),
        F.countDistinct(F.col("customer_id")).alias("unique_customers"),
        F.avg(F.col("subtotal")).alias("avg_order_value"),
        F.sum(F.col("quantity")).alias("total_units_sold") 
    )

# TODO: Add calculated metrics
daily_sales_summary = daily_sales_summary \
    .withColumn("revenue_per_customer", F.col("total_revenue") / F.col("unique_customers")) \
    .withColumn("units_per_order", F.col("total_units_sold") / F.col("order_count")) 


# SQL Version of the above 

# spark.sql("""
#          CREATE OR REPLACE TEMPORARY VIEW daily_sales_summary AS
#          SELECT 
#          daily_sales.sale_date,
#          daily_sales.category,
#          daily_sales.total_revenue,
#          daily_sales.order_count,
#          daily_sales.unique_customers,
#          daily_sales.avg_order_value,
#          daily_sales.total_units_sold,
#          (daily_sales.total_revenue /daily_sales.unique_customers) AS revenue_per_customer,
#          (daily_sales.total_units_sold / daily_sales.order_count) AS units_per_order
#          FROM
#                (
#                SELECT
#                    CAST(order_timestamp AS DATE) AS sale_date,
#                    category,
#                    SUM(subtotal) AS total_revenue,
#                    COUNT(order_id) AS order_count,
#                    COUNT(DISTINCT customer_id) AS unique_customers,
#                    AVG(subtotal) AS avg_order_value,
#                    SUM(quantity) AS total_units_sold
#                FROM SALES_SILVER_TABLE
#                GROUP BY CAST(order_timestamp AS DATE), category
#                ) daily_sales
#          """)

display(daily_sales_summary.orderBy("sale_date", "category"))

sale_date,category,total_revenue,order_count,unique_customers,avg_order_value,total_units_sold,revenue_per_customer,units_per_order
2026-05-14,null,358500.75,476,317,753.152836,1420,1130.917192429,2.9831932773109244
2026-05-14,BOOKS,354784.38,483,325,734.543230,1441,1091.644246154,2.9834368530020705
2026-05-14,Books,5399473.64,6964,3035,775.340844,20917,1779.068744646,3.0035898908673175
2026-05-14,CLOTHING,406966.93,529,360,769.313667,1582,1130.463694444,2.9905482041587903
2026-05-14,Clothing,30009377.14,39306,6528,763.480821,118224,4597.024684436,3.0077850709815297
2026-05-14,ELECTRONICS,894541.01,1203,758,743.591862,3614,1180.133258575,3.0041562759767246
2026-05-14,Electronics,14173783.97,18534,5029,764.745008,55612,2818.410015908,3.0005395489370885
2026-05-14,HOME & GARDEN,967675.65,1211,737,799.071552,3727,1312.992740841,3.077621800165153
2026-05-14,Home & Garden,8560988.45,11244,4039,761.382822,33598,2119.581195841,2.988082532906439
2026-05-14,SPORTS & OUTDOORS,636229.60,798,509,797.280201,2358,1249.959921415,2.954887218045113


---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: Daily Sales Summary

# daily_sales_summary = sales_silver \
#     .withColumn("sale_date", F.to_date("order_timestamp")) \
#     .groupBy("sale_date", "category") \
#     .agg(
#         F.sum("total_amount").alias("total_revenue"),
#         F.count("order_id").alias("order_count"),
#         F.countDistinct("customer_id").alias("unique_customers"),
#         F.avg("total_amount").alias("avg_order_value"),
#         F.sum("quantity").alias("total_units_sold"),
#     )

# daily_sales_summary = daily_sales_summary \
#     .withColumn("revenue_per_customer", F.col("total_revenue") / F.col("unique_customers")) \
#     .withColumn("units_per_order", F.col("total_units_sold") / F.col("order_count"))

# display(daily_sales_summary.orderBy("sale_date", "category"))

---
### 🎯 EXERCISE 2: Add Rolling Window Calculations
**Your task**: Calculate 7-day and 30-day rolling averages for trend analysis.

**Requirements:**
1. Create window specs:
- 7-day: Partition by category, order by sale_date, rows -6 to 0
- 30-day: Partition by category, order by sale_date, rows -29 to 0
2. Calculate rolling metrics:
- `revenue_7day_avg`: 7-day average revenue
- `revenue_30day_avg`: 30-day average revenue
- `orders_7day_total`: 7-day total orders
- `orders_30day_total`: 30-day total orders
**Window pattern:**
```python
window = Window.partitionBy("col").orderBy("col").rowsBetween(-N, 0)
df.withColumn("rolling_avg", F.avg("col").over(window))
```
**Hint**: `.rowsBetween(-6, 0)` includes current row + 6 previous = 7 total

In [0]:
# TODO: Add rolling window calculations

# Create window specifications
window_7day = Window.partitionBy("category").orderBy("sale_date").rowsBetween(-6, 0)

window_30day = Window.partitionBy("category").orderBy("sale_date").rowsBetween(-29, 0)

# Calculate rolling metrics
daily_sales_enriched = daily_sales_summary \
    .withColumn("revenue_7day_avg", F.avg(F.col("total_revenue")).over(window_7day)) \
    .withColumn("revenue_30day_avg", F.avg(F.col("total_revenue")).over(window_30day)) \
    .withColumn("orders_7day_total", F.sum(F.col("order_count")).over(window_7day)) \
    .withColumn("orders_30day_total", F.sum(F.col("order_count")).over(window_30day))

# SQL version of the above

# daily_sales_summary.dftoTemporaryView("daily_sales_summary")

#spark.sql("""
#          CREATE OR REFRESH TEMPORARY VIEW daily_sales_enriched AS

#          SELECT  
#          sale_date,
#          category,
#          total_revenue,
#          order_count, 
#          unique_customers, 
#          avg_order_value, 
#          total_units_sold, 
#          revenue_per_customer, 
#          units_per_order,
#          AVG(total_revenue) OVER (PARTITION BY category ORDER BY sale_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS revenue_7day_avg,
#          AVG(total_revenue) OVER (PARTITION BY category ORDER BY sale_date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS revenue_30day_avg,
#          SUM(order_count) OVER (PARTITION BY category ORDER BY sale_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS orders_7day_total,
#          SUM(order_count) OVER (PARTITION BY category ORDER BY sale_date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS orders_30day_total
#          FROM daily_sales_summary;
#          """)

display(daily_sales_enriched.orderBy("sale_date", "category").limit(20))

sale_date,category,total_revenue,order_count,unique_customers,avg_order_value,total_units_sold,revenue_per_customer,units_per_order,revenue_7day_avg,revenue_30day_avg,orders_7day_total,orders_30day_total
2026-05-14,null,358500.75,476,317,753.152836,1420,1130.917192429,2.9831932773109244,358500.750000,358500.750000,476,476
2026-05-14,BOOKS,354784.38,483,325,734.543230,1441,1091.644246154,2.9834368530020705,354784.380000,354784.380000,483,483
2026-05-14,Books,5399473.64,6964,3035,775.340844,20917,1779.068744646,3.0035898908673175,5399473.640000,5399473.640000,6964,6964
2026-05-14,CLOTHING,406966.93,529,360,769.313667,1582,1130.463694444,2.9905482041587903,406966.930000,406966.930000,529,529
2026-05-14,Clothing,30009377.14,39306,6528,763.480821,118224,4597.024684436,3.0077850709815297,30009377.140000,30009377.140000,39306,39306
2026-05-14,ELECTRONICS,894541.01,1203,758,743.591862,3614,1180.133258575,3.0041562759767246,894541.010000,894541.010000,1203,1203
2026-05-14,Electronics,14173783.97,18534,5029,764.745008,55612,2818.410015908,3.0005395489370885,14173783.970000,14173783.970000,18534,18534
2026-05-14,HOME & GARDEN,967675.65,1211,737,799.071552,3727,1312.992740841,3.077621800165153,967675.650000,967675.650000,1211,1211
2026-05-14,Home & Garden,8560988.45,11244,4039,761.382822,33598,2119.581195841,2.988082532906439,8560988.450000,8560988.450000,11244,11244
2026-05-14,SPORTS & OUTDOORS,636229.60,798,509,797.280201,2358,1249.959921415,2.954887218045113,636229.600000,636229.600000,798,798


---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: Rolling Window Calculations

# window_7day = Window.partitionBy("category").orderBy("sale_date").rowsBetween(-6, 0)
# window_30day = Window.partitionBy("category").orderBy("sale_date").rowsBetween(-29, 0)

# daily_sales_enriched = daily_sales_summary \
#     .withColumn("revenue_7day_avg", F.avg("total_revenue").over(window_7day)) \
#     .withColumn("revenue_30day_avg", F.avg("total_revenue").over(window_30day)) \
#     .withColumn("orders_7day_total", F.sum("order_count").over(window_7day)) \
#     .withColumn("orders_30day_total", F.sum("order_count").over(window_30day))

# display(daily_sales_enriched.orderBy("sale_date", "category").limit(20))

In [0]:
# Write to Gold table with date partitioning
daily_sales_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("sale_date") \
    .option("overwriteSchema", "true") \
    .saveAsTable(DAILY_SALES_SUMMARY_TABLE)

print(f"✅ Created Gold table: {DAILY_SALES_SUMMARY_TABLE}")

✅ Created Gold table: cert_prep_catalog.03_gold.daily_sales_summary


## Section 3: Customer Lifetime Value with RFM Segmentation
Calculate comprehensive customer metrics.
### What is RFM?
**RFM Segmentation** scores customers on three dimensions:
| Metric | Question | Score Range |
|--------|----------|-------------|
| **Recency** | How recently did they buy? | 1-5 (5 = most recent) |
| **Frequency** | How often do they buy? | 1-5 (5 = most frequent) |
| **Monetary** | How much do they spend? | 1-5 (5 = highest spend) |

**Combined RFM score**: 3-15 (sum of R+F+M)

### Customer Segments
- **Champions** (13-15): Best customers
- **Loyal** (11-12): Regular buyers
- **Potential Loyalists** (9-10): Growing engagement
- **At Risk** (7-8): Declining activity
- **Hibernating** (5-6): Inactive
- **Lost** (3-4): Haven't returned

---
### 🎯 EXERCISE 3: Calculate Customer Lifetime Value
**Your task**: Aggregate customer purchase history and calculate time-based metrics.

**Part 1 - Aggregations:**
Group by customer and calculate:
- `total_spend`: Sum of total_amount
- `total_orders`: Count of order_id
- `avg_order_value`: Average of total_amount
- `first_purchase_date`: Min of order_timestamp
- `last_purchase_date`: Max of order_timestamp
- `total_items_purchased`: Sum of quantity
- `categories_purchased`: Count distinct category

**Part 2 - Time Metrics:**
Add calculated columns:
- `days_since_first_purchase`: Days between first purchase and today
- `days_since_last_purchase`: Days between last purchase and today (Recency!)
- `customer_tenure_days`: Days between first and last purchase
**Functions:**
```python
F.datediff(F.current_date(), F.to_date("timestamp_col"))
```

In [0]:
# TODO: Calculate customer lifetime value metrics

customer_ltv = sales_silver \
    .groupBy("customer_id", "customer_name", "loyalty_tier") \
    .agg(
        # TODO: Add aggregations
        F.sum(F.col("subtotal")).alias("total_spend"),
        F.count(F.col("order_id")).alias("total_orders"),
        F.avg(F.col("subtotal")).alias("avg_order_value"),
        F.min(F.col("order_timestamp")).alias("first_purchase_date"),
        F.max(F.col("order_timestamp")).alias("last_purchase_date"),
        F.sum(F.col("quantity")).alias("total_items_purchased"),
        F.countDistinct(F.col("category")).alias("categories_purchased")    
    )

# TODO: Add time-based metrics
customer_ltv = customer_ltv \
    .withColumn("days_since_first_purchase", F.datediff(F.current_date(), F.to_date(F.col("first_purchase_date")))) \
    .withColumn("days_since_last_purchase", F.datediff(F.current_date(), F.to_date(F.col("last_purchase_date")))) \
    .withColumn("customer_tenure_days", F.datediff(F.to_date(F.col("last_purchase_date")), F.to_date(F.col("first_purchase_date"))))

# SQL version of the above:

#spark.sql("""
#          CREATE OR REPLACE TEMPORARY VIEW customer_ltv AS

#          SELECT 
#          sales.customer_name,
#          sales.customer_id,
#          sales.loyalty_tier,
#          sales.total_spend,
#          sales.total_orders,
#          sales.avg_order_value,
#          sales.first_purchase_date,
#          sales.last_purchase_date,
#          sales.total_items_purchased,
#          sales.categories_purchased,
#          DATEDIFF(CURRENT_DATE(), CAST(first_purchase_date AS DATE)) AS days_since_first_purchase,
#          DATEDIFF(CURRENT_DATE(), CAST(last_purchase_date AS DATE)) AS days_since_last_purchase,
#          DATEDIFF(CAST(last_purchase_date AS DATE), CAST(first_purchase_date AS DATE)) AS customer_tenure_days 
#          FROM 
#                (
#                SELECT
#                customer_id,
#                customer_name,
#                loyalty_tier,
#                SUM(subtotal) AS total_spend,
#                COUNT(order_id) AS total_orders,
#                AVG(subtotal) AS avg_order_value,
#                MIN(order_timestamp) AS first_purchase_date,
#                MAX(order_timestamp) AS last_purchase_date,
#                SUM(quantity) AS total_items_purchased,
#                COUNT(DISTINCT category) AS categories_purchased
#                FROM cert_prep_catalog.02_silver.sales_clean
#                GROUP BY customer_id, customer_name, loyalty_tier
#                ) sales
#          """)

display(customer_ltv.orderBy(F.desc("total_spend")).limit(20))

customer_id,customer_name,loyalty_tier,total_spend,total_orders,avg_order_value,first_purchase_date,last_purchase_date,total_items_purchased,categories_purchased,days_since_first_purchase,days_since_last_purchase,customer_tenure_days
6f75784b-a4c8-4758-8680-e3c78df5f666,Jane Johnson,null,23238774.49,30239,768.503406,2026-05-14T00:00:38,2026-05-14T23:59:38,90931,15,14,14,0
6b83606b-34ea-447b-bae3-ea7b935b6d47,null,Bronze,52891.94,63,839.554603,2026-05-14T01:37:56,2026-05-14T23:31:45,193,12,14,14,0
28aae1a8-7902-42f0-a491-660815d0ce33,Sarah Martinez,Bronze,45051.75,49,919.423469,2026-05-14T09:02:05,2026-05-14T20:36:08,149,9,14,14,0
4624ed76-a9a8-450a-8c0a-3d83feb26875,Richard Smith,null,44683.05,60,744.717500,2026-05-14T06:00:01,2026-05-14T19:53:28,182,10,14,14,0
859bf1b4-ffc9-407a-a8f5-fcd47be8bd5d,Patricia Martinez,Gold,44644.28,47,949.878298,2026-05-14T00:56:07,2026-05-14T22:14:58,151,10,14,14,0
1f6eeb1a-a024-45d7-8ac6-6bb7175926c6,Patricia Gonzalez,Silver,43337.37,55,787.952182,2026-05-14T08:49:56,2026-05-14T19:26:27,168,10,14,14,0
f4a5b80b-8911-4961-b51b-8d80ed3b648d,David Rodriguez,Gold,43302.49,48,902.135208,2026-05-14T05:26:30,2026-05-14T20:06:35,153,9,14,14,0
98ceec25-c578-4250-a850-8c196b890d1e,Thomas Martinez,Silver,42373.34,55,770.424364,2026-05-14T05:37:04,2026-05-14T19:42:10,152,11,14,14,0
dd883044-f2f7-4f94-90a6-e5cca6d4a21d,null,null,42207.67,52,811.685962,2026-05-14T05:27:52,2026-05-14T21:37:02,162,11,14,14,0
2ac02b43-61a7-4424-b95a-a6ea2bec62bf,William Smith,null,41727.57,51,818.187647,2026-05-14T06:10:19,2026-05-14T20:16:02,165,10,14,14,0


---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: Customer Lifetime Value

# customer_ltv = sales_silver \
#     .groupBy("customer_id", "customer_name", "loyalty_tier") \
#     .agg(
#         F.sum("total_amount").alias("total_spend"),
#         F.count("order_id").alias("total_orders"),
#         F.avg("total_amount").alias("avg_order_value"),
#         F.min("order_timestamp").alias("first_purchase_date"),
#         F.max("order_timestamp").alias("last_purchase_date"),
#         F.sum("quantity").alias("total_items_purchased"),
#         F.countDistinct("category").alias("categories_purchased")
#     )

# customer_ltv = (customer_ltv
#     .withColumn("days_since_first_purchase", 
#                 F.datediff(F.current_date(), F.to_date("first_purchase_date")))
#     .withColumn("days_since_last_purchase",
#                 F.datediff(F.current_date(), F.to_date("last_purchase_date")))
#     .withColumn("customer_tenure_days",
#                 F.datediff(F.col("last_purchase_date"), F.col("first_purchase_date")))
#     )

# display(customer_ltv.orderBy(F.desc("total_spend")).limit(20))

---
### 🎯 EXERCISE 4: Calculate RFM Scores and Segments
**Your task**: Create RFM scores using ntile() and assign segments.

**Part 1 - RFM Scores:**
Use `F.ntile(5)` to create 5 buckets (quintiles):
- `recency_score`: REVERSE of days_since_last_purchase (lower days = better)
- Use: `6 - F.ntile(5).over(Window.orderBy(F.desc("days_since_last_purchase")))`
- `frequency_score`: Based on total_orders (higher = better)
- Use: `F.ntile(5).over(Window.orderBy("total_orders"))`
- `monetary_score`: Based on total_spend (higher = better)
- Use: `F.ntile(5).over(Window.orderBy("total_spend"))`
- `rfm_score`: Sum of R + F + M

**Part 2 - Segments:**
Create `customer_segment` based on RFM score:
- '>= 13': Champions
- '>= 11': Loyal Customers
- '>= 9': Potential Loyalists
- '>= 7': At Risk
- '>= 5': Hibernating
- 'else': Lost

**Hint**: `ntile(5)` splits data into 5 equal-sized buckets.

In [0]:
# TODO: Calculate RFM scores using ntile

customer_ltv_rfm = customer_ltv \
    .withColumn("recency_score", 
                # TODO: 6 minus ntile of days_since_last_purchase (desc)
                6 - F.ntile(5).over(Window.orderBy(F.desc(F.col("days_since_last_purchase"))))) \
    .withColumn("frequency_score",
                # TODO: ntile of total_orders
                F.ntile(5).over(Window.orderBy(F.col("total_orders")))) \
    .withColumn("monetary_score",
                # TODO: ntile of total_spend
                F.ntile(5).over(Window.orderBy(F.col("total_spend"))))

# TODO: Calculate combined RFM score
customer_ltv_rfm = customer_ltv_rfm \
    .withColumn("rfm_score",   # TODO: sum of R + F + M
                F.col("recency_score") + F.col("frequency_score") + F.col("monetary_score"))

# SQL version of the above 

#spark.sql("""
#          
#          CREATE OR REPLACE TEMPORARY VIEW customer_ltv_rfm AS

#          SELECT 
#          cust_ltv.customer_id,
#          cust_ltv.customer_name,
#          cust_ltv.loyalty_tier,
#          cust_ltv.total_spend,
#          cust_ltv.total_orders,
#          cust_ltv.avg_order_value,
#          cust_ltv.first_purchase_date,
#          cust_ltv.last_purchase_date,
#          cust_ltv.total_items_purchased,
#          cust_ltv.categories_purchased,
#          cust_ltv.days_since_first_purchase,
#          cust_ltv.days_since_last_purchase,
#          cust_ltv.customer_tenure_days,
#          cust_ltv.recency_score,
#          cust_ltv.frequency_score,
#          cust_ltv.monetary_score,
#          cust_ltv.recency_score + cust_ltv.frequency_score + cust_ltv.monetary_score AS rfm_score
#          FROM 
#                (
#                SELECT 
#                customer_id,
#                customer_name,
#                loyalty_tier,
#                total_spend,
#                total_orders,
#                avg_order_value,
#                first_purchase_date,
#                last_purchase_date,
#                total_items_purchased,
#                categories_purchased,
#                days_since_first_purchase,
#                days_since_last_purchase,
#                customer_tenure_days,
#                6 - NTILE(5) OVER (ORDER BY days_since_last_purchase DESC) AS recency_score,
#                NTILE(5) OVER (ORDER BY total_orders) AS frequency_score,
#                NTILE(5) OVER (ORDER BY total_spend) AS monetary_score
#                FROM customer_ltv
#                ) cust_ltv
#          """)

display(customer_ltv_rfm.orderBy(F.desc("rfm_score")).limit(20))

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,customer_name,loyalty_tier,total_spend,total_orders,avg_order_value,first_purchase_date,last_purchase_date,total_items_purchased,categories_purchased,days_since_first_purchase,days_since_last_purchase,customer_tenure_days,recency_score,frequency_score,monetary_score,rfm_score
9f70811b-0844-4d2a-86d4-b2d9cfae9924,Emily Miller,Gold,11981.23,20,599.061500,2026-05-14T08:01:20,2026-05-14T18:21:41,49,6,14,14,0,5,5,5,15
73e6a6cd-f66c-45a6-ad38-cb45429cfd9d,John Brown,Silver,11925.68,16,745.355000,2026-05-14T06:03:01,2026-05-14T21:13:25,57,7,14,14,0,5,5,5,15
3147991a-e4c7-40e6-9cc6-6d15f63f5a7e,Emily Garcia,null,11753.55,19,618.607895,2026-05-14T06:53:57,2026-05-14T15:28:07,58,7,14,14,0,5,5,5,15
4d644142-3d20-4771-a3cd-dbf8389c32c8,James Lopez,Gold,12046.01,19,634.000526,2026-05-14T07:30:42,2026-05-14T17:47:23,54,5,14,14,0,5,5,5,15
1b685dd1-792f-4dc7-8bc0-a8aba656284b,Emily Gonzalez,Bronze,11862.73,16,741.420625,2026-05-14T08:47:25,2026-05-14T23:11:08,50,5,14,14,0,5,5,5,15
db560fa3-e8f7-4adf-b21d-7689102a8042,Lisa Smith,null,11907.03,19,626.685789,2026-05-14T07:19:07,2026-05-14T16:42:16,46,7,14,14,0,5,5,5,15
8c38e97e-a343-4721-8d6b-e2722b1f3b9c,James Miller,Silver,11490.05,16,718.128125,2026-05-14T10:01:25,2026-05-14T20:33:38,37,8,14,14,0,5,5,5,15
a631ea03-7962-4a40-a6f2-20311e9c278b,Jennifer Gonzalez,Gold,11714.42,16,732.151250,2026-05-14T11:31:48,2026-05-14T22:32:09,50,6,14,14,0,5,5,5,15
4b8cfb0d-993c-48c0-a0b5-979a5d96ef2a,Mary Davis,Gold,12009.68,18,667.204444,2026-05-14T10:03:14,2026-05-14T15:57:01,56,8,14,14,0,5,5,5,15
1b3773c0-3df6-42fb-a758-202ed61e699c,Thomas Brown,null,11771.29,16,735.705625,2026-05-14T10:30:44,2026-05-14T18:54:05,44,5,14,14,0,5,5,5,15


---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: RFM Scores

# customer_ltv_rfm = customer_ltv \
#     .withColumn("recency_score", 
#                 6 - F.ntile(5).over(Window.orderBy(F.desc("days_since_last_purchase")))) \
#     .withColumn("frequency_score",
#                 F.ntile(5).over(Window.orderBy("total_orders"))) \
#     .withColumn("monetary_score",
#                 F.ntile(5).over(Window.orderBy("total_spend")))

# customer_ltv_rfm = customer_ltv_rfm \
#     .withColumn("rfm_score", 
#                 F.col("recency_score") + F.col("frequency_score") + F.col("monetary_score"))

# display(customer_ltv_rfm.orderBy(F.desc("rfm_score")).limit(20))

In [0]:
# Create customer segments based on RFM score
customer_ltv_segmented = customer_ltv_rfm \
    .withColumn("customer_segment",
        F.when(F.col("rfm_score") >= 13, "Champions")
         .when(F.col("rfm_score") >= 11, "Loyal Customers")
         .when(F.col("rfm_score") >= 9, "Potential Loyalists")
         .when(F.col("rfm_score") >= 7, "At Risk")
         .when(F.col("rfm_score") >= 5, "Hibernating")
         .otherwise("Lost")
    ) \
    .withColumn("ltv_category",
        F.when(F.col("total_spend") >= 10000, "High Value")
         .when(F.col("total_spend") >= 5000, "Medium Value")
         .otherwise("Low Value")
    )

display(customer_ltv_segmented.orderBy(F.desc("rfm_score")).limit(20))

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,customer_name,loyalty_tier,total_spend,total_orders,avg_order_value,first_purchase_date,last_purchase_date,total_items_purchased,categories_purchased,days_since_first_purchase,days_since_last_purchase,customer_tenure_days,recency_score,frequency_score,monetary_score,rfm_score,customer_segment,ltv_category
9f70811b-0844-4d2a-86d4-b2d9cfae9924,Emily Miller,Gold,11981.23,20,599.061500,2026-05-14T08:01:20,2026-05-14T18:21:41,49,6,14,14,0,5,5,5,15,Champions,High Value
73e6a6cd-f66c-45a6-ad38-cb45429cfd9d,John Brown,Silver,11925.68,16,745.355000,2026-05-14T06:03:01,2026-05-14T21:13:25,57,7,14,14,0,5,5,5,15,Champions,High Value
3147991a-e4c7-40e6-9cc6-6d15f63f5a7e,Emily Garcia,null,11753.55,19,618.607895,2026-05-14T06:53:57,2026-05-14T15:28:07,58,7,14,14,0,5,5,5,15,Champions,High Value
4d644142-3d20-4771-a3cd-dbf8389c32c8,James Lopez,Gold,12046.01,19,634.000526,2026-05-14T07:30:42,2026-05-14T17:47:23,54,5,14,14,0,5,5,5,15,Champions,High Value
1b685dd1-792f-4dc7-8bc0-a8aba656284b,Emily Gonzalez,Bronze,11862.73,16,741.420625,2026-05-14T08:47:25,2026-05-14T23:11:08,50,5,14,14,0,5,5,5,15,Champions,High Value
db560fa3-e8f7-4adf-b21d-7689102a8042,Lisa Smith,null,11907.03,19,626.685789,2026-05-14T07:19:07,2026-05-14T16:42:16,46,7,14,14,0,5,5,5,15,Champions,High Value
8c38e97e-a343-4721-8d6b-e2722b1f3b9c,James Miller,Silver,11490.05,16,718.128125,2026-05-14T10:01:25,2026-05-14T20:33:38,37,8,14,14,0,5,5,5,15,Champions,High Value
a631ea03-7962-4a40-a6f2-20311e9c278b,Jennifer Gonzalez,Gold,11714.42,16,732.151250,2026-05-14T11:31:48,2026-05-14T22:32:09,50,6,14,14,0,5,5,5,15,Champions,High Value
4b8cfb0d-993c-48c0-a0b5-979a5d96ef2a,Mary Davis,Gold,12009.68,18,667.204444,2026-05-14T10:03:14,2026-05-14T15:57:01,56,8,14,14,0,5,5,5,15,Champions,High Value
1b3773c0-3df6-42fb-a758-202ed61e699c,Thomas Brown,null,11771.29,16,735.705625,2026-05-14T10:30:44,2026-05-14T18:54:05,44,5,14,14,0,5,5,5,15,Champions,High Value


In [0]:
# View segment distribution
display(
    customer_ltv_segmented.groupBy("customer_segment", "ltv_category")
    .agg(
        F.count("*").alias("customer_count"),
        F.sum("total_spend").alias("segment_revenue"),
        F.avg("total_spend").alias("avg_customer_value")
    )
    .orderBy(F.desc("segment_revenue"))
    .limit(10)
)

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_segment,ltv_category,customer_count,segment_revenue,avg_customer_value
Champions,High Value,1510,51642674.37,34200.446603
Loyal Customers,Medium Value,706,5314050.39,7526.983555
Potential Loyalists,Medium Value,664,4453700.98,6707.380994
Loyal Customers,High Value,273,3376627.93,12368.600476
At Risk,Low Value,1009,2985868.16,2959.235045
Hibernating,Low Value,1163,2414970.18,2076.500585
Potential Loyalists,Low Value,599,2285416.29,3815.386127
Champions,Medium Value,255,2117676.63,8304.614235
At Risk,Medium Value,196,1131347.31,5772.180153
Lost,Low Value,1132,1128799.97,997.173118


In [0]:
# Write to Gold table
customer_ltv_segmented.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(CUSTOMER_LTV_TABLE)

print(f"✅ Created Gold table: {CUSTOMER_LTV_TABLE}")

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ Created Gold table: cert_prep_catalog.03_gold.customer_lifetime_value


## Section 4: Product Performance with Ranking
Analyze product performance using window functions.
### Ranking Functions
| Function | Behavior | Use Case |
|----------|----------|----------|
| `RANK()` | Gaps in ranking for ties | Competition ranking |
| `DENSE_RANK()` | No gaps for ties | Consecutive ranks |
| `ROW_NUMBER()` | Unique numbers, no ties | Pagination |
**Example:**
```
Revenue: [100, 100, 90, 80]
RANK():       1,   1,  3,  4  (gap at 2)
DENSE_RANK(): 1,   1,  2,  3  (no gap)
ROW_NUMBER(): 1,   2,  3,  4  (arbitrary order for ties)
```

In [0]:
# Calculate product performance metrics
product_performance = sales_silver \
    .groupBy("product_id", "product_name", "category", "subcategory") \
    .agg(
        F.sum("subtotal").alias("total_revenue"),
        F.sum("quantity").alias("total_units_sold"),
        F.count("order_id").alias("total_orders"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.avg("unit_price").alias("avg_selling_price"),
    )

product_performance = product_performance \
    .withColumn("revenue_per_unit", F.col("total_revenue") / F.col("total_units_sold")) \
    .withColumn("orders_per_customer", F.col("total_orders") / F.col("unique_customers"))

display(product_performance.orderBy(F.desc("total_revenue")).limit(20))

product_id,product_name,category,subcategory,total_revenue,total_units_sold,total_orders,unique_customers,avg_selling_price,revenue_per_unit,orders_per_customer
P-1850,Classic Sweater,Clothing,Kids,15504572.47,61095,20259,5286,254.056379,253.778091006,3.8325766174801363
P-1209,The Complete Science,Books,null,381205.52,1439,483,329,262.437412,264.910020848,1.4680851063829787
P-1218,Modern Mirror,Home & Garden,null,377486.17,1404,473,319,267.383066,268.864793447,1.4827586206896552
P-1215,Classic Hat,Clothing,Accessories,368269.23,1468,498,336,251.755382,250.864598093,1.4821428571428572
P-1212,Rustic Clock,Home & Garden,Decor,366828.03,1339,451,323,266.590399,273.956706497,1.3962848297213621
P-1206,Elegant Table,Home & Garden,Kitchen,366614.18,1414,468,329,256.549679,259.274526167,1.4224924012158056
P-1204,Wireless Ultimate Speaker,Electronics,Mobile,364373.13,1397,472,325,260.867331,260.825433071,1.4523076923076923
P-1219,Classic Scarf,Clothing,Men,364181.79,1352,445,304,267.473034,269.365229290,1.4638157894736843
P-1211,Smart Tablet,ELECTRONICS,Audio,362774.40,1459,485,318,249.326784,248.645921864,1.5251572327044025
P-1208,Deluxe Yoga Mat,Sports & Outdoors,Cycling,352549.58,1372,480,336,256.094583,256.960335277,1.4285714285714286


---
### 🎯 EXERCISE 5: Add Product Rankings and Market Share
**Your task**: Rank products within categories and calculate market share.

**Part 1 - Ranking:**
Add columns:
- `revenue_rank_in_category`: RANK() partitioned by category, ordered by revenue desc
- `revenue_row_number`: ROW_NUMBER() with same partition and order

**Part 2 - Market Share:**
Add columns:
- `category_total_revenue`: Sum of revenue within category
- `market_share_pct`: (product revenue / category total) * 100
- `cumulative_market_share`: Running sum of market_share_pct

**Patterns:**
```python
# Ranking
window = Window.partitionBy("category").orderBy(F.desc("revenue"))
df.withColumn("rank", F.rank().over(window))
# Cumulative sum
.rowsBetween(Window.unboundedPreceding, Window.currentRow)
```

In [0]:
# TODO: Add ranking within category

category_window = Window.partitionBy(   # TODO: partition by category, order by revenue desc \
    "category").orderBy(F.desc(F.col("total_revenue")))

product_performance_ranked = product_performance \
    .withColumn("revenue_rank_in_category",   # TODO: use F.rank() \
        F.rank().over(category_window)) \
    .withColumn("revenue_row_number",   # TODO: use F.row_number() \
        F.row_number().over(category_window))

# SQL version of the above:

# product_performance.createOrReplaceTempView("product_performance")

# spark.sql("""
#          CREATE OR REPLACE TEMPORARY VIEW product_performance_ranked AS

#          SELECT 
#          product_id,
#          product_name,
#          category,
#          subcategory,
#          total_revenue,
#          total_units_sold,
#          total_orders,
#          unique_customers,
#          avg_selling_price,
#          revenue_per_unit,
#          orders_per_customer,
#          RANK() over (PARTITION BY category ORDER BY total_revenue DESC) AS revenue_rank_in_category,
#          ROW_NUMBER() over (PARTITION BY category ORDER BY total_revenue DESC) AS revenue_row_number
#          FROM product_performance

#          """)

display(product_performance_ranked.filter("revenue_rank_in_category <= 10").orderBy("category", "revenue_rank_in_category"))

product_id,product_name,category,subcategory,total_revenue,total_units_sold,total_orders,unique_customers,avg_selling_price,revenue_per_unit,orders_per_customer,revenue_rank_in_category,revenue_row_number
P-1312,null,null,null,144741.04,564,192,129,246.772344,256.633049645,1.4883720930232558,1,1
P-1534,null,null,null,61959.65,248,81,58,248.531852,249.837298387,1.396551724137931,2,2
P-1501,null,null,null,56699.79,259,83,63,223.926386,218.918108108,1.3174603174603174,3,3
P-1638,null,null,null,54301.27,208,68,48,252.243235,261.063798077,1.4166666666666667,4,4
P-1840,null,null,null,20696.75,67,27,15,282.002593,308.906716418,1.8,5,5
P-1875,null,null,null,20102.25,74,25,21,268.380800,271.652027027,1.1904761904761905,6,6
P-1300,The Advanced History,BOOKS,Non-Fiction,168980.35,690,235,160,244.074979,244.899057971,1.46875,1,1
P-1395,Guide to Adventure,BOOKS,Comics,107251.61,430,140,104,244.816786,249.422348837,1.3461538461538463,2,2
P-1565,Guide to History,BOOKS,null,45041.09,194,63,40,250.411429,232.170567010,1.575,3,3
P-1679,Fiction: A Story,BOOKS,Non-Fiction,33511.33,127,45,33,273.990444,263.868740157,1.3636363636363635,4,4


---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: Product Ranking

# category_window = Window.partitionBy("category").orderBy(F.desc("total_revenue"))

# product_performance_ranked = product_performance \
#     .withColumn("revenue_rank_in_category", F.rank().over(category_window)) \
#     .withColumn("revenue_row_number", F.row_number().over(category_window))

# display(product_performance_ranked.filter("revenue_rank_in_category <= 10").orderBy("category", "revenue_rank_in_category"))

In [0]:
# Calculate market share within category
category_totals_window = Window.partitionBy("category")

product_performance_share = (
    product_performance_ranked 
    .withColumn("category_total_revenue", F.sum("total_revenue").over(category_totals_window)) 
    .withColumn("market_share_pct", 
                (F.col("total_revenue") / F.col("category_total_revenue") * 100))
    .withColumn("cumulative_market_share",
                F.sum("market_share_pct").over(
                    Window.partitionBy("category").orderBy(F.desc("total_revenue"))
                    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
                ))
    )

display(product_performance_share.filter("revenue_rank_in_category <= 5").orderBy("category", "revenue_rank_in_category"))

product_id,product_name,category,subcategory,total_revenue,total_units_sold,total_orders,unique_customers,avg_selling_price,revenue_per_unit,orders_per_customer,revenue_rank_in_category,revenue_row_number,category_total_revenue,market_share_pct,cumulative_market_share
P-1312,null,null,null,144741.04,564,192,129,246.772344,256.633049645,1.4883720930232558,1,1,358500.75,40.373990,40.373990
P-1534,null,null,null,61959.65,248,81,58,248.531852,249.837298387,1.396551724137931,2,2,358500.75,17.282990,57.656980
P-1501,null,null,null,56699.79,259,83,63,223.926386,218.918108108,1.3174603174603174,3,3,358500.75,15.815810,73.472790
P-1638,null,null,null,54301.27,208,68,48,252.243235,261.063798077,1.4166666666666667,4,4,358500.75,15.146770,88.619560
P-1840,null,null,null,20696.75,67,27,15,282.002593,308.906716418,1.8,5,5,358500.75,5.773140,94.392700
P-1300,The Advanced History,BOOKS,Non-Fiction,168980.35,690,235,160,244.074979,244.899057971,1.46875,1,1,354784.38,47.629030,47.629030
P-1395,Guide to Adventure,BOOKS,Comics,107251.61,430,140,104,244.816786,249.422348837,1.3461538461538463,2,2,354784.38,30.230080,77.859110
P-1565,Guide to History,BOOKS,null,45041.09,194,63,40,250.411429,232.170567010,1.575,3,3,354784.38,12.695340,90.554450
P-1679,Fiction: A Story,BOOKS,Non-Fiction,33511.33,127,45,33,273.990444,263.868740157,1.3636363636363635,4,4,354784.38,9.445550,100.000000
P-1209,The Complete Science,Books,null,381205.52,1439,483,329,262.437412,264.910020848,1.4680851063829787,1,1,5399473.64,7.060050,7.060050


In [0]:
# Identify performance tiers
product_performance_final = product_performance_share \
    .withColumn("performance_tier",
        F.when(F.col("revenue_rank_in_category") <= 5, "Top Performer")
         .when(F.col("revenue_rank_in_category") <= 20, "Strong Performer")
         .when(F.col("cumulative_market_share") <= 80, "Average Performer")
         .otherwise("Underperformer")
    )

display(product_performance_final.orderBy("category", "revenue_rank_in_category").limit(50))

product_id,product_name,category,subcategory,total_revenue,total_units_sold,total_orders,unique_customers,avg_selling_price,revenue_per_unit,orders_per_customer,revenue_rank_in_category,revenue_row_number,category_total_revenue,market_share_pct,cumulative_market_share,performance_tier
P-1312,null,null,null,144741.04,564,192,129,246.772344,256.633049645,1.4883720930232558,1,1,358500.75,40.373990,40.373990,Top Performer
P-1534,null,null,null,61959.65,248,81,58,248.531852,249.837298387,1.396551724137931,2,2,358500.75,17.282990,57.656980,Top Performer
P-1501,null,null,null,56699.79,259,83,63,223.926386,218.918108108,1.3174603174603174,3,3,358500.75,15.815810,73.472790,Top Performer
P-1638,null,null,null,54301.27,208,68,48,252.243235,261.063798077,1.4166666666666667,4,4,358500.75,15.146770,88.619560,Top Performer
P-1840,null,null,null,20696.75,67,27,15,282.002593,308.906716418,1.8,5,5,358500.75,5.773140,94.392700,Top Performer
P-1875,null,null,null,20102.25,74,25,21,268.380800,271.652027027,1.1904761904761905,6,6,358500.75,5.607310,100.000010,Strong Performer
P-1300,The Advanced History,BOOKS,Non-Fiction,168980.35,690,235,160,244.074979,244.899057971,1.46875,1,1,354784.38,47.629030,47.629030,Top Performer
P-1395,Guide to Adventure,BOOKS,Comics,107251.61,430,140,104,244.816786,249.422348837,1.3461538461538463,2,2,354784.38,30.230080,77.859110,Top Performer
P-1565,Guide to History,BOOKS,null,45041.09,194,63,40,250.411429,232.170567010,1.575,3,3,354784.38,12.695340,90.554450,Top Performer
P-1679,Fiction: A Story,BOOKS,Non-Fiction,33511.33,127,45,33,273.990444,263.868740157,1.3636363636363635,4,4,354784.38,9.445550,100.000000,Top Performer


In [0]:
# Write to Gold table
product_performance_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(PRODUCT_PERFORMANCE_TABLE)

print(f"✅ Created Gold table: {PRODUCT_PERFORMANCE_TABLE}")

✅ Created Gold table: cert_prep_catalog.03_gold.product_performance


## Section 6: Optimize with Z-Ordering
**Z-ordering** co-locates related data for faster queries.
### When to Use
| Optimization | Best For | Example |
|--------------|----------|--------|
| **Partitioning** | Low cardinality, time-series | date, region |
| **Z-ordering** | High cardinality, frequent filters | customer_id, product_id, category |
### Pattern
```sql
OPTIMIZE table_name ZORDER BY (column1, column2)
```
**Best practice**: Partition by date, Z-order by frequently queried columns.

In [0]:
# Optimize daily sales (already partitioned by sale_date)
display(spark.sql(f"""
    OPTIMIZE {DAILY_SALES_SUMMARY_TABLE}
    ZORDER BY (category)
"""))

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 1, List(minCubeSize(107374182400), List(0, 0), List(1, 4934), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1779993701117, 1779993702450, 8, 0, null, List(0, 0), null, 13, 13, 0, 0, null, null)"


In [0]:
# Optimize customer LTV by segment
display(spark.sql(f"""
    OPTIMIZE {CUSTOMER_LTV_TABLE}
    ZORDER BY (customer_segment, ltv_category)
"""))

#print(f"✅ Optimized {CUSTOMER_LTV_TABLE}")

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 336919), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1779993724795, 1779993725458, 8, 0, null, List(0, 0), null, 19, 19, 0, 0, null, null)"


In [0]:
# Optimize product performance by category
display(spark.sql(f"""
    OPTIMIZE {PRODUCT_PERFORMANCE_TABLE}
    ZORDER BY (category, performance_tier)
"""))

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 35902), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1779993737240, 1779993737837, 8, 0, null, List(0, 0), null, 17, 17, 0, 0, null, null)"


In [0]:
# View optimization details
# Take a look at "partitionColumns", "clustering Columns", "numFiles", "sizeInBytes", etc
display(spark.sql(f"DESCRIBE DETAIL {DAILY_SALES_SUMMARY_TABLE}"))

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,702c9339-d157-4848-b1a2-372722f29796,cert_prep_catalog.03_gold.daily_sales_summary,null,,2026-05-24T19:40:48.977Z,2026-05-28T18:40:11.000Z,List(sale_date),List(),1,4934,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


### Additional Optimization Commands
**VACUUM** - Remove old file versions:
```sql
VACUUM table_name RETAIN 168 HOURS  -- Keep 7 days for time travel
```
**ANALYZE** - Update statistics:
```sql
ANALYZE TABLE table_name COMPUTE STATISTICS
```

In [0]:
# Best Practice: VACUUM to remove old file versions
# Removes files not required by versions older than retention threshold
# Default retention is 7 days (168 hours) - balance time travel vs storage costs

spark.sql(f"""
    VACUUM {DAILY_SALES_SUMMARY_TABLE} RETAIN 168 HOURS
"""
)
print(f"✅ Cleaned old files from {DAILY_SALES_SUMMARY_TABLE}")



✅ Cleaned old files from cert_prep_catalog.03_gold.daily_sales_summary


In [0]:
# Best Practice: ANALYZE TABLE for query optimization
# Updates table statistics used by cost-based optimizer (CBO)
display(spark.sql(f"""
    ANALYZE TABLE {CUSTOMER_LTV_TABLE} COMPUTE STATISTICS
"""
))

# Note: VACUUM permanently deletes files - cannot time travel beyond retention period

## Section 7: Summary and Checkpoint
### 🎯 Key Concepts Covered
**1. Gold Layer Design**
- Business-level aggregations
- Denormalized for query performance
- Pre-calculated metrics
**2. Advanced Aggregations**
- GROUP BY with multiple agg functions
- Rolling window calculations
- Time-series summaries
**3. Window Functions**
- `RANK()`, `ROW_NUMBER()`, `DENSE_RANK()`
- `NTILE()` for quintile scoring
- Cumulative calculations
- Partitioned windows
**4. Business Analytics**
- Customer lifetime value (LTV)
- RFM segmentation
- Product performance ranking
- Market share analysis
- Funnel conversion rates
**5. Optimization**
- Partitioning (low cardinality)
- Z-ordering (high cardinality)
- OPTIMIZE and VACUUM
### ✅ Exam Checklist
Can you:
- [ ] Write complex GROUP BY with multiple aggregations?
- [ ] Use RANK(), ROW_NUMBER(), NTILE()? 
- [ ] Create rolling windows with rowsBetween?
- [ ] Calculate conversion rates and percentages?
- [ ] Optimize tables with ZORDER BY?
- [ ] Explain partitioning vs Z-ordering?
- [ ] Implement cumulative calculations?
---
**🎉 Notebook Complete!**
The Gold layer is complete with optimized analytics tables ready for BI tools and dashboards. Proceed to Notebook 05.